## Image filters

We will load in a focus ion beam-scanning electronic microscopy (FIB-SEM) image of a solid-oxide full cell electrode.
we will try several common image filters to denoise this SEM image ("AIST_LSCF_300h_055.tif").

`https://raw.githubusercontent.com/huichiayu/cmse_202_802/main/MSE590/AIST_LSCF_300h_055.tif`

#### <p style="text-align: right;"> &#9989; **put your name here** </p>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from imageio import imread

img = imread("AIST_LSCF_300h_055.tif")

print(img.shape)

plt.figure(figsize=(10,5))
plt.imshow(img, cmap='gray')
plt.axis('off')
plt.show()

I_norm = img/255


&#9989; Do This -- What is the resolution of this image? How do you verify whether this image is grayscale?

**Below we go over several different commonly used image filters to clean the SEM image.**

### 1. Gaussian filter

A Gaussian filter smooths an image by replacing each pixel with a weighted average of nearby pixels, where closer pixels receive larger weights according to a Gaussian distribution. The 1D Gaussian function is:

$$ G(x) = \frac{1}{\sqrt{2\pi\sigma^2}} e^{-x^2/(2\sigma^2)}.$$


In image processing, we use a 2D Gaussian kernel:

$$G(x,y) = \frac{1}{\sqrt{2\pi\sigma^2}} e^{-(x^2+y^2)/(2\sigma^2)}.$$


The parameter:

- $\sigma$ controls the blur amount.
- larger $\sigma \rightarrow$ stronger smoothing.

A Gaussian filter computes a weighted local average:

$$I_{filtered}(x,y) = \int \int G(x-x',y-y')I(x',y') dx' dy'.$$

Gaussian filtering resembles:
- diffusion
- heat spreading
In fact, applying Gaussian filtering is mathematically equivalent to solving the diffusion equation for short time:

$$\frac{\partial u}{\partial t} = D\nabla^2 u.$$

**Try different value of $\sigma$.**

In [ ]:
from scipy.ndimage import gaussian_filter


img2 = gaussian_filter(I_norm, sigma=2)

plt.imshow(img2,cmap='gray')
plt.show()

### 2. Median filter 

A median filter replaces each pixel with the median value of nearby pixels instead of the average. 
For a neighborhood:

$$
\begin{bmatrix}
2 & 3 & 4 \\
5 & 100 & 6 \\
7 & 8 & 9
\end{bmatrix}
$$

Sort the values:

$$2~ 3~ 4~ 5~ 6~ 7~ 8~ 9~ 100$$

Median: $6$. So the center pixel becomes: $100\rightarrow 6$. **How it works?** $\rightarrow$ Averages are strongly affected by outliers.

**Try different size.**

In [ ]:
from scipy.ndimage import median_filter

img2 = median_filter(I_norm, size=(10,10))
plt.imshow(img2,cmap='gray')
plt.show()

### 3. Bilateral filter

A bilateral filter is an edge-preserving denoiser. Unlike Gaussian filtering, it does NOT average all nearby pixels equally.
Instead, it gives large weights only to pixels that are:
- spatially nearby
- AND have similar intensity

**Intuitive idea**

- Gaussian filter says: Average nearby pixels
- Bilateral filter says: Average nearby pixels ONLY if they look similar

The spatial Gaussian is:
$$G_s(d) = e^{-d^2/(2\sigma_s^2))}$$
The intensity Gaussian is:
$$G_s(\Delta I) = e^{-\Delta I^2/(2\sigma_r^2))}$$

**Try different values of `sigma_color` and `sigma_spatial`.**

In [ ]:
from skimage.restoration import denoise_bilateral

img2 = denoise_bilateral(
    I_norm,
    sigma_color=0.05,
    sigma_spatial=3)


plt.imshow(img2, cmap="gray")
plt.show()

### 4. Non-local means filter

Non-local means (NLM) works by finding similar image patches instead of simply averaging nearby pixels. The key idea is: Pixels with similar surrounding patterns should have similar true values even if they are far apart. Good for images often contain repeated structures.

- Non-local means filter says: Average pixels whose surrounding patches look similar.

$$w(p,q) = \exp \bigg( - \frac{||P(p)-P(q)||^2}{h^2} \bigg)$$.

where $P(p)$: patch around pixel $p$, $P(q)$: patch around pixel $q$, and $h$: filtering strength. Then filtered value:

$$I_{new}(p) = \frac{\sum_q w(p,q) I(q)}{\sum_q w(p,q)}.$$

**Try different $h$.**

In [ ]:
from skimage.restoration import denoise_nl_means

img2 = denoise_nl_means(
    I_norm,
    h=0.8*np.std(I_norm),
    fast_mode=True)

plt.imshow(img2, cmap="gray")
plt.show()

### 5. Total variation filter

`denoise_tv_chambolle` performs total variation (TV) denoising, which smooths noise while preserving sharp edges.
The key philosophy is: "Natural images are mostly smooth regions separated by sharp edges." Instead of minimizing pixel intensity directly, TV denoising minimizes the total variation of the image.

Mathematically, it solves:

$$\min_u \bigg[ \frac{1}{2} || u - f||^2 + \lambda TV(u) \bigg]$$.

where $f$: noisy image, $u$ denoised image, $\lambda$ corresponds to weight, first term: stay close to original image, second term: enforce smoothness. The total variation is 

$$TV(u) = \int |\nabla u |dx.$$

TV denoising tries to reduce unnecessary gradients while preserving major edges. So:

- smooth regions become flatter
- sharp interfaces remain sharp

TV filter: flatten smooth regions but keep boundaries. 

**Try different values of weight.**

In [ ]:
from skimage.restoration import denoise_tv_chambolle

img2 = denoise_tv_chambolle(I_norm, weight=0.1)

plt.imshow(img2, cmap="gray")
plt.show()

### 6. Wavelet filter

`denoise_wavelet` removes noise by transforming the image into wavelet coefficients, shrinking the small coefficients that are likely noise, then transforming back.

Main idea: image $\rightarrow$ wavelet transform $\rightarrow$ threshold small coefficients $\rightarrow$ inverse transform $\rightarrow$ denoised image.

Wavelets separate the image into:
- large-scale smooth features
- small-scale edges/textures/noise
  
**Try different filter parameters.**

In [ ]:
from skimage.restoration import denoise_wavelet

img2 = denoise_wavelet(I_norm)

# img2 = denoise_wavelet(
#     I_norm,
#     method='BayesShrink',
#     mode='soft',
#     wavelet='db2',
#     rescale_sigma=True,
#     channel_axis=None)

plt.imshow(img2,cmap="gray")
plt.show()

### Block-Matching and 3D filtering

It is one of the most influential classical image denoising algorithms ever developed.

The key idea is: Find similar image patches, group them together, and denoise them collaboratively.

In [ ]:
# !pip install bm3d
from bm3d import bm3d

img2 = bm3d(I_norm, sigma_psd=0.05)

plt.imshow(img2,cmap="gray")
plt.show()

### Rolling ball

The rolling ball algorithm is a classic image-processing method for background subtraction. 

&#9989; Do This -- Search what `rolling_ball` is and give a brief description of how it works.



&#9989; Do This -- Test `rolling_bal` and test different parameters to see whether this method performs better than those used earlier.


In [ ]:
# put your code here.



**Great, please upload your completed assigment to this [Link](https://www.dropbox.com/request/zkl424dlvo7s47exn7hv)**.